# Otimização Multi-objetivo

- Algoritmos offline (NSGA-II, DR-NSGA-II, Prob-MOEA/D, UA-SA-NSGA2): usam landscape do surrogate (n_var=2, n_obj=2)
- Algoritmos online (K-RVEA, KTA2, ParEGO): usam `NoisyProblem` wrapper para buscar sobre fitness ruidosas
- Todos os resultados são re-avaliados com a função verdadeira para comparação justa

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning

from src import problems as _problems_mod
from src.processing import (
    find_pareto_front,
    find_pareto_front_nd,
    create_landscape_from_predictions,
    gerar_ruido_raw,
    NoisyProblem,
)
from src.visualization import (
    display_pareto_fronts_catalog,
    display_decision_space,
)
from src.nsgaII import run_my_nsga2
from src.ua_sa_nsgaII import run_my_uasa_nsga2
from src.dr_nsgaII import run_dr_nsga2, create_landscape_for_dr_nsga2
from src.k_rvea_online_runner import run_k_rvea_online
from src.parego_runner import run_parego
from src.prob_moead_runner import run_prob_moead
from src.kta2_runner import run_kta2
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['lines.linewidth'] = 0.5

# ═══════════════════════════════════════════════════════════════════
#  Constants
# ═══════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════
#  Random seed global (reprodutibilidade)
# ═══════════════════════════════════════════════════════════════════
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROBLEM_NAMES = [
    'MMF1', 'MMF4', 'MMF11_L', 'MMF16_L3', 'MMF16_20',
    'ZDT1', 'ZDT3', 'ZDT4', 'ZDT6',
    'DTLZ1', 'DTLZ2', 'DTLZ3', 'DTLZ4', 'DTLZ7',
    'WFG1', 'WFG2', 'WFG4', 'WFG5', 'WFG9',
]

THREE_OBJ = {
    'MMF16_L3', 'MMF16_20', 'DTLZ1', 'DTLZ2', 'DTLZ3', 'DTLZ4', 'DTLZ7',
}

OFFLINE_ALGOS = {'NSGA2', 'NSGA2_surrogate', 'DR-NSGA-II', 'Prob-MOEA/D', 'UA-SA-NSGA2'}

# Controla quais algoritmos são exibidos nas visualizações e com qual cor.
# Comente/descomente linhas para mostrar/esconder algoritmos.
dict_display = {
    'KTA2':            'fuchsia',
    'ParEGO':          'lime',
    'NSGA2_surrogate': 'yellow',
    'UA-SA-NSGA2':     'cyan',
#   'NSGA2':           'forestgreen',
#   'K-RVEA':          'cyan',
#   'DR-NSGA-II':      'lime',
#   'Prob-MOEA/D':     'green',
}

# ═══════════════════════════════════════════════════════════════════
#  Algorithm selection (parametrised)
# ═══════════════════════════════════════════════════════════════════

dict_algorithms = {
    'NSGA2':           False,
    'NSGA2_surrogate': True,
    'K-RVEA':          True,
    'DR-NSGA-II':      True,
    'ParEGO':          True,
    'Prob-MOEA/D':     True,
    'KTA2':            True,
    'UA-SA-NSGA2':     True,
}

dict_algorithms_3obj = {
    'NSGA2':           False,
    'NSGA2_surrogate': True,
    'K-RVEA':          True,
    'DR-NSGA-II':      True,
    'ParEGO':          True,
    'Prob-MOEA/D':     True,
    'KTA2':            True,
    'UA-SA-NSGA2':     False,
}

# Se True, tenta carregar df_{name}_otimizacao.parquet ao invés de executar.
# Se o arquivo não existir, executa a otimização e salva.
# Se False, sempre executa a otimização.
carrega_resultados = True

# ═══════════════════════════════════════════════════════════════════
#  Modo Rápido (quick-test flag)
# ═══════════════════════════════════════════════════════════════════
# Quando True, escala para baixo os parâmetros que controlam o número
# de iterações de cada algoritmo, multiplicando-os por fator_modo_rapido.
# Parâmetros escalados (1 por algoritmo):
#   n_generations  → NSGA2, NSGA2_surrogate, DR-NSGA-II, Prob-MOEA/D, UA-SA-NSGA2
#   krvea_feval    → K-RVEA   (equivalente a FEmax)
#   kta2_feval     → KTA2     (equivalente a FEmax)
#   parego_feval   → ParEGO   (equivalente a FEmax)
MODO_RAPIDO = True
fator_modo_rapido = 0.1



# ═══════════════════════════════════════════════════════════════════
#  Noise configuration (same as NB2)
# ═══════════════════════════════════════════════════════════════════

noise_config = {
    1:  {'dist': 'bimodal',     'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'p': [0.6, 0.4], 'mu1': -2, 'sd1': 0.5, 'mu2': 2, 'sd2': 1}},
    2:  {'dist': 'student_t',   'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'df': 3}},
    3:  {'dist': 'lognormal',   'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'mean': 0, 'sigma': 0.8}},
    4:  {'dist': 'poisson',     'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'lam': 5.0}},
    5:  {'dist': 'binomial',    'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'n': 10, 'p': 0.5}},
    6:  {'dist': 'geometric',   'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'p': 0.3}},
    7:  {'dist': 'exponential', 'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'scale': 1.0}},
    8:  {'dist': 'uniform',     'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'low': -1.0, 'high': 1.0}},
    9:  {'dist': 'laplace',     'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'loc': 0.0, 'scale': 1.0}},
    10: {'dist': 'gamma',       'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'shape': 2.0, 'scale': 2.0}},
    11: {'dist': 'beta',        'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'a': 0.5, 'b': 0.5}},
    12: {'dist': 'rayleigh',    'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'scale': 1.0}},
    13: {'dist': 'weibull',     'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'a': 1.5}},
    14: {'dist': 'logistic',    'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'loc': 0.0, 'scale': 1.0}},
    15: {'dist': 'pareto',      'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {'a': 3.0}},
    16: {'dist': 'rademacher',  'target_mean': 0.0, 'forca_ruido': 0.2, 'params': {}},
}

# ═══════════════════════════════════════════════════════════════════
#  Base config (NSGA-II parameters, adapted per problem)
# ═══════════════════════════════════════════════════════════════════

BASE_CONFIG = {
    'population_size': 100,
    'n_generations': 100,   # NSGA2 / NSGA2_surrogate / DR-NSGA-II / Prob-MOEA/D / UA-SA-NSGA2
    'krvea_feval':   300,   # K-RVEA  (equivalente a FEmax)
    'kta2_feval':    300,   # KTA2    (equivalente a FEmax)
    'parego_feval':  250,   # ParEGO  (equivalente a FEmax)
    'k_tournament': 2,
    'crossover_prob': 0.9,
    'crossover_eta': 15,
    'mutation_eta': 20,
    'seed': 42,
    'track_progress': True,
    'utiliza_ds_niching': True,
    'pesos_ds_niching': [1, 1],
    'maximize': False,
    'n_restricoes': 0,
    'tipo_variavel_genotipo': float,
    'verbose': False,
    'mostrar_grafico': False,
}


# Parâmetros de BASE_CONFIG escalados pelo MODO_RAPIDO (1 por algoritmo):
#   n_generations → NSGA2, NSGA2_surrogate, DR-NSGA-II, Prob-MOEA/D, UA-SA-NSGA2
#   krvea_feval   → K-RVEA
#   kta2_feval    → KTA2
#   parego_feval  → ParEGO
indicadores = ['n_generations', 'krvea_feval', 'kta2_feval', 'parego_feval']

if MODO_RAPIDO:
    for indicador in indicadores:
        BASE_CONFIG[indicador] = max(1, int(BASE_CONFIG[indicador] * fator_modo_rapido))

# ═══════════════════════════════════════════════════════════════════
#  Helpers
# ═══════════════════════════════════════════════════════════════════

def train_kriging_surrogates(df_noisy, x_cols, f_cols, n_train=500, seed=42):
    """Train one Kriging (GP) model per objective on noisy data.

    This replaces CatBoost for NSGA2_surrogate, DR-NSGA-II and Prob-MOEA/D,
    aligning with the papers that assume Kriging surrogates.
    """
    df_train = df_noisy.sample(n=min(n_train, len(df_noisy)), random_state=seed)
    models = []
    for fc in f_cols:
        kernel = ConstantKernel(1.0) * Matern(nu=2.5)
        gp = GaussianProcessRegressor(
            kernel=kernel, alpha=1e-3,
            n_restarts_optimizer=2, normalize_y=True)
        gp.fit(df_train[x_cols].values, df_train[fc].values)
        models.append(gp)
    return models


def _instantiate_problem(name):
    return getattr(_problems_mod, name)()


def _safe_algo_filename(algo_name):
    return algo_name.replace('/', '_').replace(' ', '_').replace('-', '_').lower()


def _reevaluate_clean(df_pareto, clean_problem):
    """Overwrite f columns with true (clean) fitness values."""
    n_var = clean_problem.n_var
    n_obj = clean_problem.n_obj
    x_cols = [f'x_{i+1}' for i in range(n_var)]
    f_cols = [f'f{j+1}' for j in range(n_obj)]
    if len(df_pareto) == 0 or not all(c in df_pareto.columns for c in x_cols):
        for fc in f_cols:
            if fc not in df_pareto.columns:
                df_pareto[fc] = pd.Series(dtype=float)
        return df_pareto
    X = df_pareto[x_cols].values
    out = {'F': np.zeros((len(X), n_obj))}
    clean_problem._evaluate(X, out)
    for j, fc in enumerate(f_cols):
        df_pareto[fc] = out['F'][:, j]
    return df_pareto


# ═══════════════════════════════════════════════════════════════════
#  Metrics (HV, IGD, Gamma, Delta)
# ═══════════════════════════════════════════════════════════════════

def compute_hypervolume(F, ref_point):
    """Hypervolume indicator (S-metric).

    Used in: K-RVEA, DR-NSGA-II, ParEGO, Prob-MOEA/D (4 papers).
    Measures volume of objective space dominated by F, bounded by ref_point.
    LARGER is better.

    For 2-obj: incremental rectangle method.
    For 3+ obj: uses pymoo's HV implementation.
    """
    from pymoo.indicators.hv import HV
    ind = HV(ref_point=ref_point)
    return ind(F)


def compute_igd(F_approx, F_true):
    """Inverted Generational Distance.

    Used in: K-RVEA, KTA2 (2 papers).
    IGD = (1/|P*|) * sum_{p in P*} min_{a in A} ||p - a||
    Measures how well A covers the true PF. LOWER is better.
    """
    from scipy.spatial.distance import cdist
    D = cdist(F_true, F_approx)
    return np.mean(np.min(D, axis=1))


def compute_gamma(F_approx, F_true):
    """Gamma convergence metric (Deb et al., 2002 — NSGA-II paper, Sec. IV-B).

    Gamma = (1/|A|) * sum_{a in A} min_{p in P*} ||a - p||
    Measures how close obtained solutions are to the true PF.
    LOWER is better (better convergence).
    """
    from scipy.spatial.distance import cdist
    D = cdist(F_approx, F_true)
    return np.mean(np.min(D, axis=1))


def compute_delta(F_approx, F_true):
    """Delta diversity metric (Deb et al., 2002 — NSGA-II paper, Sec. IV-B).

    Measures uniformity of spread along the obtained front.
    Delta = (d_f + d_l + sum|d_i - d_bar|) / (d_f + d_l + (N-1)*d_bar)
    LOWER is better (more uniform distribution).

    For 3-obj problems, uses a simplified version based on consecutive
    distances after sorting by first objective (the paper's original
    formulation is for 2-obj but the principle generalises).
    """
    if len(F_approx) < 2:
        return float('inf')
    from scipy.spatial.distance import cdist
    n_obj = F_approx.shape[1]
    idx_sort = np.lexsort(F_approx[:, ::-1].T)
    F_sorted = F_approx[idx_sort]
    d = np.array([np.linalg.norm(F_sorted[i+1] - F_sorted[i])
                   for i in range(len(F_sorted) - 1)])
    if len(d) == 0:
        return float('inf')
    d_bar = d.mean()
    D_to_true = cdist(F_true, F_approx)
    extremes_true_idx = []
    for j in range(n_obj):
        extremes_true_idx.append(np.argmin(F_true[:, j]))
    d_f = np.min(D_to_true[extremes_true_idx[0]])
    d_l = np.min(D_to_true[extremes_true_idx[-1]])
    denom = d_f + d_l + (len(d)) * d_bar
    if denom == 0:
        return 0.0
    return (d_f + d_l + np.sum(np.abs(d - d_bar))) / denom


def evaluate_metrics(F_approx, problem, n_pf_points=500):
    """Compute all 4 metrics for an approximation set.

    Returns dict with keys: HV, IGD, Gamma, Delta.
    """
    _, F_true = problem.true_pareto_front(n_pf_points)
    ref_point = np.max(F_true, axis=0) * 1.1 + 0.01
    metrics = {}
    try:
        metrics['HV'] = float(compute_hypervolume(F_approx, ref_point))
    except Exception:
        metrics['HV'] = None
    metrics['IGD'] = float(compute_igd(F_approx, F_true))
    metrics['Gamma'] = float(compute_gamma(F_approx, F_true))
    metrics['Delta'] = float(compute_delta(F_approx, F_true))
    return metrics


# Global dict to accumulate results across all problems
ALL_METRICS = {}


# ═══════════════════════════════════════════════════════════════════
#  run_optimization
# ═══════════════════════════════════════════════════════════════════

def run_optimization(clean_problem, noisy_problem,
                     df_previsao, df_landscape, df_mcmc,
                     algos, config, kriging_models=None):
    """Run selected algorithms. Returns {algo_name: df_pareto} with clean f values.

    Kriging-based algorithms (NSGA2_surrogate, DR-NSGA-II, Prob-MOEA/D) use
    kriging_models for evaluation when provided — matching their respective
    papers.  UA-SA-NSGA2 keeps CatBoost/landscape.
    """
    n_var = clean_problem.n_var
    n_obj = clean_problem.n_obj
    suppress = not config.get('verbose', True)
    if suppress:
        warnings.filterwarnings('ignore', category=ConvergenceWarning)

    returns = {}

    # ── NSGA2 (original fitness, landscape-based) ─────────────────
    if algos.get('NSGA2') and n_obj == 2 and df_landscape is not None:
        df_aux = df_previsao.copy()
        for j in range(1, n_obj + 1):
            df_aux[f'fitness{j}'] = df_aux[f'f{j}_original']
        df_p, _, _ = run_my_nsga2(config, df_aux)
        for j in range(1, n_obj + 1):
            df_p[f'f{j}'] = df_p[f'f{j}_original']
        returns['NSGA2'] = df_p

    # ── NSGA2_surrogate (CatBoost landscape, same approach as UA-SA-NSGA2) ──
    if algos.get('NSGA2_surrogate') and df_previsao is not None:
        df_aux = df_previsao.copy()
        for j in range(1, n_obj + 1):
            df_aux[f'fitness{j}'] = df_aux[f'f{j}_predicted']
        df_p, _, _ = run_my_nsga2(config, df_aux)
        for j in range(1, n_obj + 1):
            if f'f{j}_original' in df_p.columns:
                df_p[f'f{j}'] = df_p[f'f{j}_original']
            else:
                _reevaluate_clean(df_p, clean_problem)
                break
        returns['NSGA2_surrogate'] = df_p

    # ── DR-NSGA-II (Kriging with native mean+std) ─────────────────
    if algos.get('DR-NSGA-II') and kriging_models is not None:
        cfg_dr = config.copy()
        df_p, _, _ = run_dr_nsga2(cfg_dr, kriging_models=kriging_models, z=1.28)
        _reevaluate_clean(df_p, clean_problem)
        returns['DR-NSGA-II'] = df_p

    # ── Prob-MOEA/D (Kriging with native mean+std for MC sampling) ─
    if algos.get('Prob-MOEA/D') and kriging_models is not None:
        cfg_pm = config.copy()
        cfg_pm['prob_moead_samples'] = 50
        cfg_pm['moead_neighborhood'] = 20
        df_p, _, _ = run_prob_moead(cfg_pm, kriging_models=kriging_models)
        _reevaluate_clean(df_p, clean_problem)
        returns['Prob-MOEA/D'] = df_p

    # ── UA-SA-NSGA2 (keeps CatBoost landscape) ────────────────────
    if algos.get('UA-SA-NSGA2') and n_obj == 2 and df_landscape is not None:
        cfg_ua = config.copy()
        cfg_ua['utiliza_ds_niching'] = False
        df_p, _ = run_my_uasa_nsga2(cfg_ua, df_landscape, save_history=True)
        land_cols = [f'x_{i+1}_landscape' for i in range(n_var)]
        df_p.drop_duplicates(subset=land_cols, inplace=True)
        rename_map = {}
        for i in range(n_var):
            rename_map[f'x_{i+1}_landscape'] = f'x_{i+1}'
        for j in range(1, n_obj + 1):
            rename_map[f'f{j}_original'] = f'f{j}'
        keep = [c for c in rename_map if c in df_p.columns]
        df_p = df_p[keep].rename(columns=rename_map)
        returns['UA-SA-NSGA2'] = df_p

    # ── Online algorithms (use noisy_problem, then re-evaluate clean) ─
    if algos.get('K-RVEA'):
        cfg_kr = config.copy()
        cfg_kr['FEmax'] = config.get('krvea_feval', 300)
        cfg_kr['krvea_wmax'] = 20
        cfg_kr['krvea_u'] = 5
        cfg_kr['krvea_NI'] = 11 * n_var - 1
        if n_obj >= 3:
            cfg_kr['population_size'] = 13
            cfg_kr['krvea_NI'] = max(cfg_kr['krvea_NI'], 100)
            cfg_kr['FEmax'] = max(cfg_kr['FEmax'], 500)
        df_p, _ = run_k_rvea_online(noisy_problem, cfg_kr)
        _reevaluate_clean(df_p, clean_problem)
        returns['K-RVEA'] = df_p

    if algos.get('ParEGO'):
        cfg_pe = config.copy()
        cfg_pe['FEmax'] = config.get('parego_feval', 250)
        cfg_pe['parego_NI'] = 11 * n_var - 1
        df_p, _ = run_parego(noisy_problem, cfg_pe)
        _reevaluate_clean(df_p, clean_problem)
        returns['ParEGO'] = df_p

    if algos.get('KTA2'):
        cfg_kt = config.copy()
        cfg_kt['FEmax'] = config.get('kta2_feval', 300)
        cfg_kt['kta2_archive_size'] = 100
        cfg_kt['kta2_eta'] = 5
        df_p, _ = run_kta2(noisy_problem, cfg_kt)
        _reevaluate_clean(df_p, clean_problem)
        returns['KTA2'] = df_p

    if suppress:
        warnings.filterwarnings('default', category=ConvergenceWarning)

    return returns


# ═══════════════════════════════════════════════════════════════════
#  Visualisation
# ═══════════════════════════════════════════════════════════════════

def visualize_optimization_results(name, problem, df_landscape_data, returns):
    """Visualize Pareto fronts from algorithms using catalog-aware functions.

    Only algorithms present in ``dict_display`` (and in ``returns``) are shown.
    """
    n_obj = problem.n_obj
    n_var = problem.n_var
    x_cols = [f'x_{i+1}' for i in range(n_var)]
    f_cols = [f'f{j+1}' for j in range(n_obj)]

    X_true, _ = problem.true_pareto_front()
    F_landscape = df_landscape_data[f_cols].values
    X_landscape = df_landscape_data[x_cols].values

    algo_names, algo_F, algo_X, algo_colors = [], [], [], []
    for algo_name, color in dict_display.items():
        if algo_name not in returns:
            continue
        df_res = returns[algo_name]
        avail_f = [c for c in f_cols if c in df_res.columns]
        avail_x = [c for c in x_cols if c in df_res.columns]
        if len(avail_f) != n_obj:
            print(f'  ⚠ {algo_name}: colunas de objetivo incompletas, pulando visualização')
            continue
        algo_names.append(algo_name)
        algo_F.append(df_res[avail_f].values)
        algo_X.append(df_res[avail_x].values)
        algo_colors.append(color)

    if not algo_names:
        print(f'  Sem resultados para visualizar em {name}')
        return

    print('\n' + '=' * 80)
    print(f'{name} — ESPAÇO DE OBJETIVOS')
    print('=' * 80)

    display_pareto_fronts_catalog(
        problem, F_landscape,
        pareto_fronts_list=algo_F,
        front_names=algo_names,
        front_colors=algo_colors,
        sample_size=100_000,
        title=f'{name} — Espaço de Objetivos',
    )

    print('\n' + '=' * 80)
    print(f'{name} — ESPAÇO DE DECISÕES')
    print('=' * 80)

    display_decision_space(
        problem, X_landscape, F_landscape,
        X_true=X_true,
        X_emp_list=algo_X,
        emp_names=algo_names,
        emp_colors=algo_colors,
        cmap='plasma',
        title=name,
    )


# ═══════════════════════════════════════════════════════════════════
#  Pipeline
# ═══════════════════════════════════════════════════════════════════

def optimization_pipeline(name, 
                          dict_algs,
                          sampling_method='sobol'
                         ):
    """Full pipeline: load/run algorithms + visualize.

    Behaviour controlled by the global ``carrega_resultados``:
    - True  → tenta carregar ``df_{name}_otimizacao.parquet``; se não existir, executa e salva.
    - False → sempre executa a otimização e salva.
    """
    print(f'\n{"#" * 80}')
    print(f'# {name}')
    print(f'{"#" * 80}')

    problem = _instantiate_problem(name)
    n_var = problem.n_var
    n_obj = problem.n_obj

    base = Path('data/dataframes') / name
    cache_path = base / f'df_{name}_otimizacao.parquet'

    # Filter algorithms that cannot run on this problem
    algos = {}
    for algo, enabled in dict_algs.items():
        if not enabled:
            continue
        if algo == 'UA-SA-NSGA2' and n_obj != 2:
            print(f'  ⚠ UA-SA-NSGA2: requer n_obj=2 (usa CatBoost/landscape) — pulando')
            continue
        algos[algo] = True

    print(f'  n_var={n_var}, n_obj={n_obj}')
    print(f'  Algoritmos aplicáveis: {list(algos.keys())}')

    # Try to load cached results
    loaded = False
    if carrega_resultados and cache_path.exists():
        df_cache = pd.read_parquet(cache_path)
        if 'otimizador' in df_cache.columns:
            returns = {}
            for algo_name in df_cache['otimizador'].unique():
                returns[algo_name] = df_cache[df_cache['otimizador'] == algo_name].drop(columns=['otimizador'])
            print(f'  Carregado: {cache_path} ({len(df_cache)} linhas, {len(returns)} algoritmos)')
            loaded = True

    if not loaded:
        # Config
        cfg = BASE_CONFIG.copy()
        cfg['tamanho_genotipo'] = n_var
        cfg['limite_inferior'] = problem.xl.copy()
        cfg['limite_superior'] = problem.xu.copy()
        cfg['n_objetivos'] = n_obj
        cfg['fitness_cols'] = [f'fitness{j+1}' for j in range(n_obj)]
        cfg['mutation_prob'] = 1.0 / n_var
        n_gens = cfg['n_generations']
        cfg['alpha_exploration_rank_distance'] = np.linspace(1.0, 0.0, n_gens)

        # Load input data
        f_cols = [f'f{j+1}' for j in range(n_obj)]
        df_previsao = pd.read_parquet(base / f'df_{name}_previsao.parquet')
        mcmc_path_file = base / f'df_{name}_mcmc.parquet'
        df_mcmc = None
        if mcmc_path_file.exists():
            df_mcmc = pd.read_parquet(mcmc_path_file)
            if 'index_linha' in df_mcmc.columns:
                df_mcmc = df_mcmc.rename(columns={'index_linha': 'id_simulacao'})
            df_mcmc.sort_values(['regiao', 'id_simulacao'], inplace=True)
            cfg['n_simulations'] = int(df_mcmc['id_simulacao'].max()) + 1

        # NoisyProblem for online runners
        df_landscape_data_full = pd.read_parquet(base / f'df_{name}_landscape_{sampling_method}.parquet')
        print('df_landscape_data_full.columns')
        print(df_landscape_data_full.columns)
        mean_f = np.array([df_landscape_data_full[c].mean() for c in f_cols])
        noisy_problem = NoisyProblem(problem, noise_config, mean_f)

        # Landscape for offline algorithms
        df_landscape = None
        if n_obj == 2 and df_mcmc is not None:
            x_cols_land = [f'x_{i+1}' for i in range(n_var)]
            df_landscape = create_landscape_from_predictions(
                df_previsao=df_previsao, df_mcmc=df_mcmc,
                decision_variables=x_cols_land)

        # Train Kriging surrogates for NSGA2_surrogate, DR-NSGA-II, Prob-MOEA/D
        f_noisy_cols = [f'f{j+1}' for j in range(n_obj)]
        x_cols_kr = [f'x_{i+1}' for i in range(n_var)]
        print('  Treinando Kriging surrogates...')
        kriging_models = train_kriging_surrogates(
            df_previsao, x_cols_kr, f_noisy_cols, n_train=500)
        print(f'  {len(kriging_models)} modelos Kriging treinados.')

        returns = run_optimization(
            problem, noisy_problem,
            df_previsao, df_landscape, df_mcmc,
            algos, cfg, kriging_models=kriging_models)

        # Save combined parquet
        parts = []
        for algo_name, df_res in returns.items():
            tmp = df_res.copy()
            tmp['otimizador'] = algo_name
            parts.append(tmp)
        if parts:
            df_combined = pd.concat(parts, ignore_index=True)
            df_combined.to_parquet(cache_path)
            print(f'  Salvo: {cache_path} ({len(df_combined)} linhas)')

    # Load landscape for visualization (always needed)
    f_cols = [f'f{j+1}' for j in range(n_obj)]
    df_landscape_data = pd.read_parquet(base / f'df_{name}_landscape_{sampling_method}.parquet')
    print('df_landscape_data.columns')
    print(df_landscape_data.columns)

    visualize_optimization_results(name, problem, df_landscape_data, returns)

    # Compute metrics for each algorithm
    print('\n' + '-' * 60)
    print(f'{name} — MÉTRICAS DE AVALIAÇÃO')
    print('-' * 60)
    f_cols = [f'f{j+1}' for j in range(problem.n_obj)]
    problem_metrics = {}
    for algo_name, df_res in returns.items():
        avail_f = [c for c in f_cols if c in df_res.columns]
        if len(avail_f) != problem.n_obj:
            problem_metrics[algo_name] = None
            continue
        F_algo = df_res[avail_f].values.astype(float)
        m = evaluate_metrics(F_algo, problem)
        problem_metrics[algo_name] = m
        print(f'  {algo_name:20s}  HV={m["HV"]:.4f}  IGD={m["IGD"]:.4f}  '
              f'Γ={m["Gamma"]:.4f}  Δ={m["Delta"]:.4f}')
    ALL_METRICS[name] = problem_metrics

    return returns

## MMF1

In [ ]:
optimization_pipeline('MMF1', dict_algorithms)

## MMF4

In [ ]:
optimization_pipeline('MMF4', dict_algorithms)

## MMF11_L

In [ ]:
optimization_pipeline('MMF11_L', dict_algorithms)

## MMF16_L3

In [ ]:
optimization_pipeline('MMF16_L3', dict_algorithms_3obj)

## MMF16_20

In [ ]:
optimization_pipeline('MMF16_20', dict_algorithms_3obj)

## ZDT1

In [ ]:
optimization_pipeline('ZDT1', dict_algorithms)

## ZDT3

In [ ]:
optimization_pipeline('ZDT3', dict_algorithms)

## ZDT4

In [ ]:
optimization_pipeline('ZDT4', dict_algorithms)

## ZDT6

In [ ]:
optimization_pipeline('ZDT6', dict_algorithms)

## DTLZ1

In [ ]:
optimization_pipeline('DTLZ1', dict_algorithms_3obj)

## DTLZ2

In [ ]:
optimization_pipeline('DTLZ2', dict_algorithms_3obj)

## DTLZ3

In [ ]:
optimization_pipeline('DTLZ3', dict_algorithms_3obj)

## DTLZ4

In [ ]:
optimization_pipeline('DTLZ4', dict_algorithms_3obj)

## DTLZ7

In [ ]:
#optimization_pipeline('DTLZ7', dict_algorithms_3obj)

## WFG1

In [ ]:
optimization_pipeline('WFG1', dict_algorithms)

## WFG2

In [ ]:
optimization_pipeline('WFG2', dict_algorithms)

## WFG4

In [ ]:
optimization_pipeline('WFG4', dict_algorithms)

## WFG5

In [ ]:
optimization_pipeline('WFG5', dict_algorithms)

## WFG9

In [ ]:
optimization_pipeline('WFG9', dict_algorithms)

## Métricas de Avaliação

As 4 métricas abaixo foram selecionadas com base na frequência de uso nos artigos dos algoritmos implementados e na cobertura de diferentes aspectos da qualidade das soluções (convergência e diversidade).

### 1. Hypervolume (HV) — Convergência + Diversidade | ↑ Maior = melhor
**Aparece em:** K-RVEA, DR-NSGA-II, ParEGO, Prob-MOEA/D (4 artigos)

Volume do espaço de objetivos **dominado** pelo conjunto de soluções A, limitado por um ponto de referência r (pior que todos os pontos). Um conjunto com melhor convergência E melhor espalhamento terá maior HV.

**Cálculo:** Para cada par de pontos consecutivos (ordenados por f₁), calcula-se a área/volume do retângulo/hipercubo dominado. HV = soma dessas áreas.

**Exemplo 2D:** A = {(0.2, 0.8), (0.5, 0.3), (0.9, 0.1)}, r = (1.1, 1.1)
- Retângulo 1: (1.1−0.2) × (1.1−0.8) = 0.27
- Retângulo 2: (1.1−0.5) × (0.8−0.3) = 0.30
- Retângulo 3: (1.1−0.9) × (0.3−0.1) = 0.04
- **HV = 0.61**

### 2. IGD (Inverted Generational Distance) — Convergência + Diversidade | ↓ Menor = melhor
**Aparece em:** K-RVEA, KTA2 (2 artigos)

Para cada ponto `p` do PF verdadeiro P*, encontra a distância ao ponto mais próximo no conjunto A.

**Cálculo:** IGD = (1/|P*|) × Σ_{p ∈ P*} min_{a ∈ A} ‖p − a‖

**Exemplo:** P* = {(0,1), (0.5,0.5), (1,0)}, A = {(0.1,0.9), (0.6,0.4)}
- d(p₁) = min(0.141, 0.781) = 0.141
- d(p₂) = min(0.566, 0.141) = 0.141
- d(p₃) = min(1.273, 0.566) = 0.566
- **IGD = 0.283**

Se A não cobre uma região do PF verdadeiro, os pontos nessa região terão distâncias altas → IGD sobe.

### 3. Gamma (Γ) — Convergência pura | ↓ Menor = melhor
**Aparece em:** NSGA-II (Deb et al., 2002, Sec. IV-B)

Inverso do IGD: para cada ponto `a` do conjunto encontrado A, distância ao PF verdadeiro mais próximo.

**Cálculo:** Γ = (1/|A|) × Σ_{a ∈ A} min_{p ∈ P*} ‖a − p‖

Se todas as soluções estão exatamente no PF, Γ = 0.

### 4. Delta (Δ) — Diversidade / Uniformidade | ↓ Menor = melhor
**Aparece em:** NSGA-II (Deb et al., 2002, Sec. IV-B)

Mede a uniformidade da distribuição de soluções ao longo do front.

**Cálculo:** Δ = (d_f + d_l + Σ|d_i − d̄|) / (d_f + d_l + (N−1)×d̄)

Onde:
- d_i = distância entre soluções consecutivas (ordenadas por f₁)
- d̄ = média das d_i
- d_f = distância do extremo do PF verdadeiro ao primeiro ponto de A
- d_l = distância do extremo do PF verdadeiro ao último ponto de A

Se todas as d_i = d̄ (perfeitamente uniforme) e d_f = d_l = 0 (cobre os extremos): **Δ = 0**.

## Sumário de Resultados

In [ ]:
# Construir tabela de métricas: linhas = problemas, colunas = algoritmos
import pandas as pd

metric_names = ['HV', 'IGD', 'Gamma', 'Delta']
all_algos = sorted(set(algo for pm in ALL_METRICS.values() for algo in pm))

rows = []
for prob_name, algo_dict in ALL_METRICS.items():
    for algo in all_algos:
        if algo in algo_dict and algo_dict[algo] is not None:
            m = algo_dict[algo]
            rows.append({
                'Problema': prob_name,
                'Algoritmo': algo,
                'HV ↑': f'{m["HV"]:.4f}' if m['HV'] is not None else '-',
                'IGD ↓': f'{m["IGD"]:.4f}',
                'Γ (conv.) ↓': f'{m["Gamma"]:.4f}',
                'Δ (div.) ↓': f'{m["Delta"]:.4f}',
            })
        else:
            rows.append({
                'Problema': prob_name,
                'Algoritmo': algo,
                'HV ↑': f'não executado',
                'IGD ↓': f'não executado',
                'Γ (conv.) ↓': f'não executado',
                'Δ (div.) ↓': f'não executado',
            })

df_summary = pd.DataFrame(rows)
df_pivot = df_summary.pivot(index='Problema', columns='Algoritmo',
                            values=['HV ↑', 'IGD ↓', 'Γ (conv.) ↓', 'Δ (div.) ↓'])

# Reorder problems
prob_order = [p for p in PROBLEM_NAMES if p in df_pivot.index]
df_pivot = df_pivot.loc[prob_order]

print('═' * 100)
print('SUMÁRIO DE MÉTRICAS — TODOS OS PROBLEMAS × ALGORITMOS')
print('═' * 100)
print('HV ↑ = Hypervolume (maior melhor)')
print('IGD ↓ = Inverted Generational Distance (menor melhor)')
print('Γ ↓ = Gamma — convergência (menor melhor)')
print('Δ ↓ = Delta — diversidade (menor melhor)')
print()

for metric in ['HV ↑', 'IGD ↓', 'Γ (conv.) ↓', 'Δ (div.) ↓']:
    print(f'\n{"=" * 80}')
    print(f'  {metric}')
    print(f'{"=" * 80}')
    sub = df_pivot[metric]
    display(sub)
